## 1. Датасеты

In [ ]:
# https://universe.roboflow.com/obscenity/adult-content-8gj6y
# https://universe.roboflow.com/computer-vision-y8bhq/no-hate-symbols
# https://universe.roboflow.com/kyunghee-university-ada5d/smoking-detection-3gefl
# https://universe.roboflow.com/adonantonin/alcohol-iaeeq
# https://universe.roboflow.com/rage-7avz1/drug-xw0hc
# https://universe.roboflow.com/pathikreet/weapons-detection-qbwdp

## 3. Объединение датасетов

Все датасеты сливаются в один с переназначением ID классов.


In [1]:
import os
import random
import shutil
import zipfile

import dotenv
import requests
import yaml

dotenv.load_dotenv(".env")
random.seed(42)

# Конфигурация
RF_API_KEY = os.environ["ROBOFLOW_API_KEY"]
SAMPLE_SIZE = 1500  # Ограничение на класс
OUTPUT_DIR = "./datasets"
CLASSES = ["adult", "hate", "smoking", "alcohol", "drugs", "weapons"]

In [7]:
datasets_config = [
    {
        "workspace": "obscenity",
        "project": "adult-content-8gj6y",
        "version": 1,
        "name": "adult",
        "target_class": "adult",
    },
    {
        "workspace": "computer-vision-y8bhq",
        "project": "no-hate-symbols",
        "version": 1,
        "name": "hate",
        "target_class": "hate",
    },
    {
        "workspace": "kyunghee-university-ada5d",
        "project": "smoking-detection-3gefl",
        "version": 4,
        "name": "smoking",
        "target_class": "smoking",
    },
    {
        "workspace": "adonantonin",
        "project": "alcohol-iaeeq",
        "version": 4,
        "name": "alcohol",
        "target_class": "alcohol",
    },
    {
        "workspace": "rage-7avz1",
        "project": "drug-xw0hc",
        "version": 2,
        "name": "drugs",
        "target_class": "drugs",
    },
    {
        "workspace": "pathikreet",
        "project": "weapons-detection-qbwdp",
        "version": 2,
        "name": "weapons",
        "target_class": "weapons",
    },
]

In [8]:
def download_roboflow_dataset(workspace, project, version, api_key, format="yolov8", output_dir="."):
    url = f"https://api.roboflow.com/{workspace}/{project}/{version}/{format}"
    params = {"api_key": api_key}

    print("  Получаем ссылку для скачивания...")
    response = requests.get(url, params=params)
    if response.status_code != 200:
        raise Exception(f"Ошибка API: {response.status_code} - {response.text}")

    data = response.json()
    download_url = data.get("export", {}).get("link")
    if not download_url:
        raise Exception(f"Не удалось получить ссылку. Ответ: {data}")

    zip_path = f"{output_dir}.zip"
    print("  Скачиваем датасет...")
    with requests.get(download_url, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    print(f"\r  {downloaded / total * 100:.1f}%", end="", flush=True)
    print()

    print("  Распаковываем...")
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(output_dir)
    os.remove(zip_path)
    return output_dir


In [9]:
def get_dataset_classes(local_path):
    """Читает классы из data.yaml датасета"""
    yaml_path = f"{local_path}/data.yaml"
    if not os.path.exists(yaml_path):
        # Ищем yaml в подпапках
        for root, dirs, files in os.walk(local_path):
            for file in files:
                if file.endswith(".yaml"):
                    yaml_path = os.path.join(root, file)
                    break

    with open(yaml_path) as f:
        data = yaml.safe_load(f)

    names = data.get("names", [])
    print(f"  Классы датасета ({len(names)}): {names}")
    return names


In [10]:
def build_class_remap(source_classes, target_classes, target_class):
    """Все классы датасета мапятся в один target_class"""
    target_id = {name.lower(): idx for idx, name in enumerate(target_classes)}.get(target_class.lower())
    
    if target_id is None:
        raise Exception(f"Класс '{target_class}' не найден в CLASSES: {target_classes}")
    
    # Все old_id → один target_id
    remap = {old_id: target_id for old_id in range(len(source_classes))}
    print(f"  Все {len(source_classes)} классов → '{target_class}' (id={target_id})")
    return remap


In [11]:
def copy_label_with_remap(src_label, dst_label, remap):
    """Копирует label-файл, переиндексируя классы согласно remap.

    Строки с классами не из remap пропускаются.
    """
    lines_out = []
    with open(src_label) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            old_class = int(parts[0])
            if old_class in remap:
                parts[0] = str(remap[old_class])
                lines_out.append(" ".join(parts))
            else:
                pass  # класс не из remap — пропускаем

    if lines_out:
        with open(dst_label, "w") as f:
            f.write("\n".join(lines_out))
        return True
    return False  # файл стал пустым — объектов нужных классов нет

In [12]:
for split in ["train", "valid", "test"]:
    os.makedirs(f"{OUTPUT_DIR}/{split}/images", exist_ok=True)
    os.makedirs(f"{OUTPUT_DIR}/{split}/labels", exist_ok=True)

for ds_info in datasets_config:
    print(f"\nЗагрузка {ds_info['name']}...")
    local_path = f"{OUTPUT_DIR}/raw_{ds_info['name']}"

    download_roboflow_dataset(
        workspace=ds_info["workspace"],
        project=ds_info["project"],
        version=ds_info["version"],
        api_key=RF_API_KEY,
        format="yolov8",
        output_dir=local_path,
    )

    # Читаем классы этого датасета и строим ремаппинг
    source_classes = get_dataset_classes(local_path)
    remap = build_class_remap(source_classes, CLASSES, ds_info["target_class"])

    print(f"  Ремаппинг классов: {remap}")

    # ── Train с сэмплированием ──
    train_images_path = f"{local_path}/train/images"
    if not os.path.exists(train_images_path):
        print(f"  ВНИМАНИЕ: {train_images_path} не найдена")
        print(f"  Содержимое {local_path}: {os.listdir(local_path)}")
        continue

    train_imgs = [f for f in os.listdir(train_images_path) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

    if len(train_imgs) > SAMPLE_SIZE:
        selected = random.sample(train_imgs, SAMPLE_SIZE)
        print(f"  Сэмплирование: {len(train_imgs)} -> {SAMPLE_SIZE}")
    else:
        selected = train_imgs
        print(f"  Берем все ({len(train_imgs)})")

    copied, skipped = 0, 0
    for img_name in selected:
        base_name = img_name.rsplit(".", 1)[0]
        src_label = f"{local_path}/train/labels/{base_name}.txt"
        dst_label = f"{OUTPUT_DIR}/train/labels/{ds_info['name']}_{base_name}.txt"

        if os.path.exists(src_label):
            ok = copy_label_with_remap(src_label, dst_label, remap)
            if ok:
                shutil.copy(f"{train_images_path}/{img_name}", f"{OUTPUT_DIR}/train/images/{ds_info['name']}_{img_name}")
                copied += 1
            else:
                skipped += 1  # label стал пустым — изображение не нужно
        # Если label нет вообще — копируем как background (без аннотаций)
        else:
            shutil.copy(f"{train_images_path}/{img_name}", f"{OUTPUT_DIR}/train/images/{ds_info['name']}_{img_name}")
            copied += 1

    print(f"  Train скопировано: {copied}, пропущено (нет нужных классов): {skipped}")

    # ── Valid и Test ──
    for split in ["valid", "test"]:
        split_img_path = f"{local_path}/{split}/images"
        if not os.path.exists(split_img_path):
            continue
        for img_name in os.listdir(split_img_path):
            if not img_name.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            base = img_name.rsplit(".", 1)[0]
            src_label = f"{local_path}/{split}/labels/{base}.txt"
            dst_label = f"{OUTPUT_DIR}/{split}/labels/{ds_info['name']}_{base}.txt"

            if os.path.exists(src_label):
                ok = copy_label_with_remap(src_label, dst_label, remap)
                if ok:
                    shutil.copy(f"{split_img_path}/{img_name}", f"{OUTPUT_DIR}/{split}/images/{ds_info['name']}_{img_name}")
            else:
                shutil.copy(f"{split_img_path}/{img_name}", f"{OUTPUT_DIR}/{split}/images/{ds_info['name']}_{img_name}")


# ── data.yaml ──
data_yaml = {
    "path": os.path.abspath(OUTPUT_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(CLASSES),
    "names": CLASSES,
}
with open(f"{OUTPUT_DIR}/data.yaml", "w") as f:
    yaml.dump(data_yaml, f)

print("\nГотово! data.yaml создан.")


Загрузка adult...
  Получаем ссылку для скачивания...
  Скачиваем датасет...
  100.0%
  Распаковываем...
  Классы датасета (5): ['breast_nude', 'breasts_seminude', 'buttock', 'genitalia_female', 'genitalia_male']
  Все 5 классов → 'adult' (id=0)
  Ремаппинг классов: {0: 0, 1: 0, 2: 0, 3: 0, 4: 0}
  Берем все (864)
  Train скопировано: 864, пропущено (нет нужных классов): 0

Загрузка hate...
  Получаем ссылку для скачивания...
  Скачиваем датасет...
  100.0%
  Распаковываем...
  Классы датасета (6): ['Antifa', 'anti-muslim', 'anti-semitic', 'isis', 'neo-nazi', 'white-supremesist']
  Все 6 классов → 'hate' (id=1)
  Ремаппинг классов: {0: 1, 1: 1, 2: 1, 3: 1, 4: 1, 5: 1}
  Сэмплирование: 5472 -> 1500
  Train скопировано: 1220, пропущено (нет нужных классов): 280

Загрузка smoking...
  Получаем ссылку для скачивания...
  Скачиваем датасет...
  100.0%
  Распаковываем...
  Классы датасета (1): ['smoke']
  Все 1 классов → 'smoking' (id=2)
  Ремаппинг классов: {0: 2}
  Сэмплирование: 17683 ->

In [13]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # Загружаем предобученные веса

results = model.train(
    data="./datasets/data.yaml",
    epochs=50,           # Для 10к картинок 50 эпох обычно достаточно
    imgsz=416,           # 416 быстрее, чем 640, и достаточно для детекции объектов
    batch=-1,            # Для 6GB VRAM начните с 8. Если OOM -> ставьте 4 или batch=-1
    device=0,            # GPU
    workers=4,           # Число потоков загрузки данных
    cache="ram",         # Кеширование в RAM ускоряет обучение (если хватает ОЗУ)
    amp=True,            # Mixed precision (автоматически ускоряет и экономит память)
    patience=5,          # Early stopping, если качество перестало расти
    optimizer="AdamW",   # Часто сходится быстрее SGD на малых batch
    lr0=0.001,           # Чуть меньший learning rate для AdamW
    close_mosaic=5,      # Отключение мозаики в последние 5 эпох для стабилизации
)

New https://pypi.org/project/ultralytics/8.4.48 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.45  Python-3.13.1 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./datasets/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train